In [1]:
import numpy as np
np.random.seed(1)

# VOCABOLARIO: 6 parole -> n feature
vocab = ['compra', 'ora', 'gratis', 'viagra', 'riunione', 'progetto']
n = len(vocab)

# probabilità VERE che ogni parola compaia, per classe (verità di laboratorio, per generare)
p_spam = np.array([0.8, 0.7, 0.6, 0.5, 0.05, 0.05])   # spam: tante parole "commerciali"
p_ham  = np.array([0.1, 0.2, 0.05, 0.01, 0.6, 0.7])   # ham:  tante parole "di lavoro"

m1, m0 = 60, 40                                        # 60 spam, 40 ham
# ogni email = vettore binario: 1 se la parola compare. (rand < p) -> presente con prob. p
X1 = (np.random.rand(m1, n) < p_spam).astype(int)      # (60, 6) email spam
X0 = (np.random.rand(m0, n) < p_ham).astype(int)       # (40, 6) email ham
X = np.vstack([X1, X0])                                # (100, 6)
y = np.array([1]*m1 + [0]*m0)                          # 1 = spam, 0 = ham
m = X.shape[0]

print('X shape:', X.shape, ' | y shape:', y.shape)
print('prima email spam:', X[0])
print('prima email ham :', X[60])

X shape: (100, 6)  | y shape: (100,)
prima email spam: [1 0 1 1 0 0]
prima email ham : [0 1 0 0 0 1]


In [2]:
# prior: frazione di spam
phi_y = np.mean(y == 1)

# per ogni parola j: frazione di email di QUELLA classe che la contengono
# Laplace: +1 al numeratore, +2 al denominatore (la parola ha 2 stati: presente/assente)
def stima_phi(Xc):
  return (Xc.sum(axis=0) + 1) / (Xc.shape[0] + 2)    # (n,) una prob. per parola

phi_j1 = stima_phi(X[y == 1])    # p(parola=1 | spam)  per ogni parola
phi_j0 = stima_phi(X[y == 0])    # p(parola=1 | ham)   per ogni parola

print('phi_y =', round(float(phi_y), 3))
print(f"{'parola':10s} {'spam':>6s} {'ham':>6s}")
for w, a, b in zip(vocab, phi_j1, phi_j0):
  print(f"{w:10s} {a:6.2f} {b:6.2f}")

phi_y = 0.6
parola       spam    ham
compra       0.85   0.10
ora          0.50   0.24
gratis       0.60   0.10
viagra       0.40   0.02
riunione     0.08   0.50
progetto     0.05   0.81


In [3]:
# PREDIZIONE in log: somma di log invece del prodotto -> niente underflow (vedi 7c)
def logprob(X, phi_j, prior):
  # log p(x_j|y): parola presente (x=1) -> log(phi) ; assente (x=0) -> log(1-phi)
  lp = X*np.log(phi_j) + (1-X)*np.log(1-phi_j)       # (m, n)
  return np.log(prior) + lp.sum(axis=1)              # (m,) log-score per email

def predici(X):
  score_spam = logprob(X, phi_j1, phi_y)             # log[ p(y=1)·∏ p(x_j|1) ]
  score_ham  = logprob(X, phi_j0, 1 - phi_y)         # log[ p(y=0)·∏ p(x_j|0) ]
  return (score_spam > score_ham).astype(int)

# accuracy sul training
print('accuracy:', round(float((predici(X) == y).mean()*100), 1), '%')

# email nuove: vettori binari sul vocabolario [compra,ora,gratis,viagra,riunione,progetto]
nuove = np.array([
  [1, 0, 1, 1, 0, 0],   # "compra gratis viagra"
  [0, 0, 0, 0, 1, 1],   # "riunione progetto"
])
for x_new, testo in zip(nuove, ['compra gratis viagra', 'riunione progetto']):
  pred = int(predici(x_new[None, :])[0])
  print(f"'{testo}' -> {'SPAM' if pred==1 else 'HAM'}")

accuracy: 98.0 %
'compra gratis viagra' -> SPAM
'riunione progetto' -> HAM
